# Workplace EV Charging: Robust Behavioral, Forecasting, and Quantum-Classical Analysis

## Google Colab / Google Drive companion notebook

This revised notebook provides an auditable end-to-end workflow for the manuscript on workplace EV charging. It is designed to:

1. mount Google Drive and locate the charging dataset;
2. create stronger behavioral and predictive results with leakage-aware validation;
3. distinguish **empirical findings** from **scenario-based control experiments**;
4. save every publication-ready figure, table, configuration file, and a human-readable `outputs_summary.txt` under:

`MyDrive/Outputs/EV_Charging_Quantum`

> **Scientific boundary.** Bunching near the four-hour threshold is evidence *consistent with* tariff response, but it does not identify a continuous price elasticity. The control section therefore reports sensitivity across assumed elasticities instead of claiming direct calibration.

### Recommended runtime

Google Colab CPU is sufficient for the classical sections. The bounded quantum benchmark is enabled by default and may take several additional minutes. Set `RUN_QUANTUM = False` only when a quick classical-only run is desired.

## 0. Environment setup and Google Drive

In [ ]:
# Install only packages that may be absent from a standard Colab runtime.
# After installation, Colab normally does not need to be restarted.
!pip -q install pennylane pennylane-lightning xgboost openpyxl tabulate pandapower

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, re, json, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, TimeSeriesSplit
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             balanced_accuracy_score, f1_score, brier_score_loss,
                             mean_absolute_error, mean_squared_error, r2_score)

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED); random.seed(SEED)
sns.set_theme(style='whitegrid', context='notebook')

DRIVE_ROOT = Path('/content/drive/MyDrive')
OUTPUT_ROOT = DRIVE_ROOT / 'Outputs' / 'EV_Charging_Quantum'
FIG_DIR = OUTPUT_ROOT / 'Figures'
TABLE_DIR = OUTPUT_ROOT / 'Tables'
MODEL_DIR = OUTPUT_ROOT / 'Models'
for directory in (OUTPUT_ROOT, FIG_DIR, TABLE_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Prevent stale tables/figures from earlier notebook versions from contaminating
# a new ZIP export. Only generated artifacts inside this notebook's output folders
# are removed; the input dataset and unrelated Drive files are never touched.
CLEAN_OLD_GENERATED_FILES = True
if CLEAN_OLD_GENERATED_FILES:
    generated_patterns = [(FIG_DIR,'Figure_*'),(TABLE_DIR,'Table_*'),(MODEL_DIR,'*')]
    removed=0
    for folder,pattern in generated_patterns:
        for old in folder.glob(pattern):
            if old.is_file(): old.unlink(); removed+=1
    for old_name in ['outputs_summary.txt','output_inventory.csv','run_configuration.json']:
        old=OUTPUT_ROOT/old_name
        if old.is_file(): old.unlink(); removed+=1
    print(f'Output hygiene: removed {removed} generated artifacts from the previous run.')

# Put the CSV in one of these preferred locations. If it is not there, the
# notebook searches MyDrive recursively for station_data_dataverse.csv.
PREFERRED_DATA_PATHS = [
    DRIVE_ROOT / 'Datasets' / 'Energy' / 'station_data_dataverse.csv',
    DRIVE_ROOT / 'Outputs' / 'EV_Charging_Quantum' / 'Input' / 'station_data_dataverse.csv',
    DRIVE_ROOT / 'Inputs' / 'EV_Charging_Quantum' / 'station_data_dataverse.csv',
    DRIVE_ROOT / 'station_data_dataverse.csv',
]

RUN_QUANTUM = True        # Produces the exploratory VQC table and convergence figure.
FAST_MODE = False         # Set True for a short smoke test.
N_BOOT = 500 if FAST_MODE else 2000
N_AGENT_SEEDS = 3 if FAST_MODE else 10
N_QUANTUM_SEEDS = 2 if FAST_MODE else 5
QUANTUM_STEPS = 10 if FAST_MODE else 25
QUANTUM_CAP_PER_SPLIT = 100 if FAST_MODE else 250

summary_lines = []
def note(text=''):
    text = str(text)
    print(text)
    summary_lines.append(text)

def save_table(frame, name, index=False):
    """Save each table in CSV and Excel formats."""
    csv_path = TABLE_DIR / f'{name}.csv'
    xlsx_path = TABLE_DIR / f'{name}.xlsx'
    frame.to_csv(csv_path, index=index)
    frame.to_excel(xlsx_path, index=index)
    print(f'Saved table: {csv_path.name} and {xlsx_path.name}')
    return csv_path

def save_figure(fig, name):
    """Save each figure as high-resolution PNG and vector PDF."""
    png = FIG_DIR / f'{name}.png'
    pdf = FIG_DIR / f'{name}.pdf'
    fig.savefig(png, dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(pdf, bbox_inches='tight', facecolor='white')
    print(f'Saved figure: {png.name} and {pdf.name}')
    return png

note('EV CHARGING QUANTUM ANALYSIS')
note('=' * 72)
note(f'Output directory: {OUTPUT_ROOT}')
note(f'Random seed: {SEED}; bootstrap replicates: {N_BOOT}')


## 1. Dataset discovery, audit, and principled cleaning

The dataset is read from Google Drive. Cleaning removes failed/zero-energy plug-ins, implausibly long sessions, and extremely short sessions. Long sessions are **not** removed with IQR rules because doing so would selectively delete the paid observations central to the tariff analysis.

In [ ]:
def locate_dataset():
    for path in PREFERRED_DATA_PATHS:
        if path.exists():
            return path
    matches = list(DRIVE_ROOT.rglob('station_data_dataverse.csv'))
    if not matches:
        raise FileNotFoundError(
            'station_data_dataverse.csv was not found in MyDrive. '
            'Expected location: MyDrive/Datasets/Energy/station_data_dataverse.csv. '
            'Upload it there and rerun this cell.'
        )
    if len(matches) > 1:
        print('Multiple datasets found; using:', matches[0])
    return matches[0]

DATA_PATH = locate_dataset()
raw = pd.read_csv(DATA_PATH)
required = {'sessionId','userId','stationId','kwhTotal','dollars','chargeTimeHrs','startTime'}
missing = required - set(raw.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

audit = {
    'data_path': str(DATA_PATH), 'raw_rows': len(raw),
    'users': raw.userId.nunique(), 'stations': raw.stationId.nunique(),
    'zero_energy': int((raw.kwhTotal <= 0).sum()),
    'distance_missing_pct': float(raw.distance.isna().mean()*100) if 'distance' in raw else np.nan,
    'maximum_duration_h': float(raw.chargeTimeHrs.max())
}

df = raw.loc[(raw.kwhTotal > 0) & raw.chargeTimeHrs.between(0.05, 24)].copy()
df = df.reset_index(drop=True)
audit['clean_rows'] = len(df)
audit['removed_rows'] = len(raw)-len(df)

audit_table = pd.DataFrame({'measure': audit.keys(), 'value': audit.values()})
save_table(audit_table, 'Table_01_Data_Audit')
note('\n1. DATA AUDIT')
for k,v in audit.items(): note(f'- {k}: {v}')
display(audit_table)


In [ ]:
# Descriptive statistics are kept separate from inferential results.
desc_cols = [c for c in ['kwhTotal','chargeTimeHrs','dollars','startTime','distance'] if c in df]
descriptive = df[desc_cols].describe(percentiles=[.05,.25,.5,.75,.95]).T.reset_index(names='variable')
save_table(descriptive, 'Table_02_Descriptive_Statistics')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(df.kwhTotal, bins=45, ax=axes[0], color='#246A8D')
axes[0].set(xlabel='Energy per session (kWh)', title='Session energy')
sns.histplot(df.chargeTimeHrs, bins=60, ax=axes[1], color='#2B7A78')
axes[1].axvline(4, color='#B0443C', linestyle='--', label='4-hour threshold')
axes[1].set(xlabel='Duration (h)', title='Session duration'); axes[1].legend()
day_cols = [c for c in ['Mon','Tues','Wed','Thurs','Fri','Sat','Sun'] if c in df]
day_counts = pd.Series({c: df[c].sum() for c in day_cols})
day_counts.plot.bar(ax=axes[2], color='#F4B942')
axes[2].set(xlabel='Day', ylabel='Sessions', title='Weekly activity')
fig.suptitle('Figure 1. Cleaned workplace-charging dataset', fontweight='bold')
fig.tight_layout(); save_figure(fig, 'Figure_01_Data_Overview'); plt.show()


## 2. Four-hour threshold analysis

The excess-mass estimator fits a polynomial counterfactual outside a prespecified bunching window. Robustness is strengthened through:

- session and user-cluster bootstraps;
- polynomial-order and window-width sensitivity;
- placebo cutoffs away from four hours;
- a local density-jump diagnostic.

These tests strengthen the evidence, but they do not eliminate all alternative explanations; results are described as **consistent with tariff response**, not definitive causal identification.

In [ ]:
def bunching_estimate(duration, cutoff=4.0, order=5, window=0.5,
                       fit_lo=1.0, fit_hi=6.0, bin_width=0.1):
    duration = np.asarray(duration, float)
    edges = np.arange(0, 8 + bin_width, bin_width)
    counts, _ = np.histogram(duration, bins=edges)
    centers = edges[:-1] + bin_width/2
    in_window = (centers >= cutoff-window) & (centers < cutoff)
    fit_mask = (centers >= fit_lo) & (centers <= fit_hi) & ~in_window
    coef = np.polyfit(centers[fit_mask], counts[fit_mask], order)
    counterfactual = np.clip(np.polyval(coef, centers), 0, None)
    excess = counts[in_window].sum() - counterfactual[in_window].sum()
    normalization = counterfactual[(centers>=fit_lo)&(centers<=fit_hi)].mean()
    b = excess / max(normalization, 1e-12)
    return {'b':b, 'excess':excess, 'centers':centers,
            'counts':counts, 'counterfactual':counterfactual}

base_b = bunching_estimate(df.chargeTimeHrs)
rng = np.random.default_rng(SEED)

# Session bootstrap
session_bs = np.array([
    bunching_estimate(rng.choice(df.chargeTimeHrs.values, len(df), replace=True))['b']
    for _ in range(N_BOOT)
])

# User-cluster bootstrap retains within-user dependence.
users = df.userId.unique()
cluster_bs = []
for _ in range(N_BOOT):
    sampled = rng.choice(users, len(users), replace=True)
    durations = np.concatenate([df.loc[df.userId.eq(u),'chargeTimeHrs'].values for u in sampled])
    cluster_bs.append(bunching_estimate(durations)['b'])
cluster_bs = np.asarray(cluster_bs)

boot_table = pd.DataFrame([
    ['Point estimate', base_b['b'], np.nan, np.nan, base_b['excess']],
    ['Session bootstrap', session_bs.mean(), *np.quantile(session_bs,[.025,.975]), np.nan],
    ['User-cluster bootstrap', cluster_bs.mean(), *np.quantile(cluster_bs,[.025,.975]), np.nan],
], columns=['estimate','b','ci_low','ci_high','excess_sessions'])
save_table(boot_table, 'Table_03_Bunching_Main_Results')
display(boot_table)

note('\n2. THRESHOLD ANALYSIS')
note(f"- Excess-mass b: {base_b['b']:.3f}; excess sessions: {base_b['excess']:.1f}")
note(f"- Session-bootstrap 95% CI: {np.quantile(session_bs,.025):.3f} to {np.quantile(session_bs,.975):.3f}")
note(f"- User-cluster-bootstrap 95% CI: {np.quantile(cluster_bs,.025):.3f} to {np.quantile(cluster_bs,.975):.3f}")


In [ ]:
# Sensitivity surface: polynomial order x bunching-window width.
sensitivity = []
for order in range(3, 7):
    for window in [0.3,0.4,0.5,0.6,0.7]:
        sensitivity.append({'order':order,'window_h':window,
                            'b':bunching_estimate(df.chargeTimeHrs, order=order, window=window)['b']})
sensitivity = pd.DataFrame(sensitivity)
save_table(sensitivity, 'Table_04_Bunching_Sensitivity')

# Placebo cutoffs use the same estimator at neighboring locations.
placebos = pd.DataFrame({'cutoff_h':np.arange(2.5,5.6,.25)})
placebos['b'] = placebos.cutoff_h.map(lambda z: bunching_estimate(df.chargeTimeHrs,cutoff=z)['b'])
save_table(placebos, 'Table_05_Placebo_Cutoffs')

# A transparent local log-density jump diagnostic (not treated as causal proof).
def local_log_jump(x, cutoff=4.0, bandwidth=.25):
    left=((x>=cutoff-bandwidth)&(x<cutoff)).sum()
    right=((x>=cutoff)&(x<cutoff+bandwidth)).sum()
    return np.log((left+.5)/(right+.5))
jump = local_log_jump(df.chargeTimeHrs.values)
jump_bs=[]
for _ in range(N_BOOT):
    sampled=rng.choice(users,len(users),replace=True)
    x=np.concatenate([df.loc[df.userId.eq(u),'chargeTimeHrs'].values for u in sampled])
    jump_bs.append(local_log_jump(x))
jump_ci=np.quantile(jump_bs,[.025,.975])
note(f'- Local log-density jump: {jump:.3f}; user-bootstrap CI {jump_ci[0]:.3f} to {jump_ci[1]:.3f}')

fig, axes = plt.subplots(2,2,figsize=(13,9))
c=base_b['centers']; obs=base_b['counts']; cf=base_b['counterfactual']
axes[0,0].bar(c,obs,width=.09,color='#246A8D',alpha=.8,label='Observed')
axes[0,0].plot(c,cf,'--',color='#B0443C',lw=2,label='Counterfactual')
axes[0,0].axvspan(3.5,4,color='#F4B942',alpha=.25); axes[0,0].axvline(4,color='k',ls=':')
axes[0,0].set(xlim=(1,6),xlabel='Duration (h)',ylabel='Sessions',title=f"Bunching estimate: b={base_b['b']:.2f}"); axes[0,0].legend()
axes[0,1].hist(session_bs,bins=35,alpha=.6,label='Session',color='#246A8D')
axes[0,1].hist(cluster_bs,bins=35,alpha=.5,label='User-cluster',color='#2B7A78')
axes[0,1].axvline(0,color='k',ls=':'); axes[0,1].set(xlabel='b',title='Bootstrap distributions'); axes[0,1].legend()
piv=sensitivity.pivot(index='order',columns='window_h',values='b')
sns.heatmap(piv,annot=True,fmt='.2f',cmap='YlGnBu',ax=axes[1,0]); axes[1,0].set_title('Specification sensitivity')
axes[1,1].plot(placebos.cutoff_h,placebos.b,'-o',color='#59636E'); axes[1,1].axvline(4,color='#B0443C',ls='--')
axes[1,1].set(xlabel='Candidate cutoff (h)',ylabel='b',title='Placebo-cutoff scan')
fig.suptitle('Figure 2. Robustness of the four-hour threshold evidence',fontweight='bold')
fig.tight_layout(); save_figure(fig,'Figure_02_Bunching_Robustness'); plt.show()


## 3. User profiling without a circular validation outcome

The original workflow clustered users using `paid_share` and then validated clusters by comparing `paid_share`, making the contrast partly definitional. Here, the primary segmentation uses **activity and charging behavior only**. Fee exposure is retained as an external outcome.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler

user = df.groupby('userId').agg(
    sessions=('sessionId','size'), mean_kwh=('kwhTotal','mean'),
    median_kwh=('kwhTotal','median'), mean_duration=('chargeTimeHrs','mean'),
    duration_cv=('chargeTimeHrs',lambda x:x.std()/x.mean() if x.mean() else 0),
    fee_share=('dollars',lambda x:(x>0).mean()), total_kwh=('kwhTotal','sum')
).fillna(0)

cluster_features=['sessions','mean_kwh','mean_duration','duration_cv']
Z=StandardScaler().fit_transform(user[cluster_features])
sil=[]
for k in range(2,7):
    labels=KMeans(k,n_init=50,random_state=SEED).fit_predict(Z)
    sil.append({'k':k,'silhouette':silhouette_score(Z,labels)})
sil=pd.DataFrame(sil); best_k=int(sil.loc[sil.silhouette.idxmax(),'k'])
user['cluster']=KMeans(best_k,n_init=50,random_state=SEED).fit_predict(Z)
# Bootstrap stability: refit on sampled users and compare predicted labels on the
# full standardized dataset. ARI is invariant to arbitrary cluster-label names.
stability=[]
for b in range(300 if not FAST_MODE else 50):
    ii=rng.choice(len(Z),len(Z),replace=True)
    boot_model=KMeans(best_k,n_init=20,random_state=SEED+b).fit(Z[ii])
    stability.append(adjusted_rand_score(user.cluster,boot_model.predict(Z)))
order=user.groupby('cluster').sessions.mean().sort_values().index
name_map={cluster:f'Activity tier {i+1}' for i,cluster in enumerate(order)}
user['segment']=user.cluster.map(name_map)

profile=user.groupby('segment').agg(users=('sessions','size'),sessions_mean=('sessions','mean'),
    mean_kwh=('mean_kwh','mean'),mean_duration=('mean_duration','mean'),fee_share=('fee_share','mean'),
    total_kwh=('total_kwh','sum')).reset_index()
save_table(sil,'Table_06_Cluster_Selection')
save_table(profile,'Table_07_User_Segment_Profiles')
# Operational alternatives are reported because the silhouette-optimal five-cluster
# solution is exploratory and contains small groups. No alternative is presented as
# uniquely "true"; the table exposes the interpretability/stability trade-off.
alternative_profiles=[]
for k in [2,3,5]:
    lab=KMeans(k,n_init=50,random_state=SEED).fit_predict(Z)
    for cluster in range(k):
        ix=lab==cluster
        alternative_profiles.append({'k':k,'cluster':cluster,'users':int(ix.sum()),
            'sessions_mean':user.sessions.values[ix].mean(),'mean_kwh':user.mean_kwh.values[ix].mean(),
            'mean_duration':user.mean_duration.values[ix].mean(),'fee_share_outcome':user.fee_share.values[ix].mean()})
alternative_profiles=pd.DataFrame(alternative_profiles)
save_table(alternative_profiles,'Table_08_Alternative_Operational_Segmentations')
note('\n3. USER PROFILING')
note(f'- Activity-based cluster count selected by silhouette: {best_k}')
note(f'- Bootstrap cluster stability ARI: median {np.median(stability):.3f}; 5th-95th percentile {np.quantile(stability,.05):.3f}-{np.quantile(stability,.95):.3f}')
note('- Fee share was excluded from the clustering inputs and evaluated only as an outcome.')
display(profile)

fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(sil.k,sil.silhouette,'-o',color='#246A8D'); axes[0].set(xlabel='k',ylabel='Silhouette',title='Cluster selection')
sns.scatterplot(data=user,x='sessions',y='mean_kwh',hue='segment',size='fee_share',sizes=(30,220),ax=axes[1])
axes[1].set(xscale='log',xlabel='Sessions (log scale)',ylabel='Mean energy (kWh)',title='Activity-based user profiles')
fig.suptitle('Figure 3. Non-circular behavioral segmentation',fontweight='bold')
fig.tight_layout(); save_figure(fig,'Figure_03_User_Segmentation'); plt.show()


## 4. Leakage-controlled session-level prediction

The target is whether session energy exceeds the cleaned-sample median. The **unseen-user** benchmark excludes any feature derived from the held-out user's outcomes. This avoids the original leakage risk created by computing `user_hist` before the user-disjoint split.

All preprocessing is fitted inside each fold. ROC-AUC, PR-AUC, balanced accuracy, F1, and Brier loss are reported.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier

df['hour_sin']=np.sin(2*np.pi*df.startTime/24)
df['hour_cos']=np.cos(2*np.pi*df.startTime/24)
df['is_weekend']=df[[c for c in ['Sat','Sun'] if c in df]].sum(axis=1).clip(0,1)
numeric=['hour_sin','hour_cos','is_weekend']
for c in ['managerVehicle']:
    if c in df: numeric.append(c)
categorical=[c for c in ['facilityType','platform'] if c in df]
features=numeric+categorical
X=df[features].copy(); y=(df.kwhTotal>df.kwhTotal.median()).astype(int); groups=df.userId

pre=ColumnTransformer([
    ('num',StandardScaler(),numeric),
    ('cat',OneHotEncoder(handle_unknown='ignore',sparse_output=False),categorical)
],remainder='drop')
models={
    'Dummy':DummyClassifier(strategy='prior'),
    'Logistic regression':LogisticRegression(max_iter=2000,class_weight='balanced'),
    'RBF-SVM':SVC(C=1,probability=True,class_weight='balanced',random_state=SEED),
    'Random forest':RandomForestClassifier(n_estimators=400,max_depth=7,min_samples_leaf=5,class_weight='balanced',random_state=SEED,n_jobs=-1),
    'Gradient boosting':GradientBoostingClassifier(random_state=SEED),
    'XGBoost':XGBClassifier(n_estimators=350,max_depth=4,learning_rate=.04,subsample=.85,colsample_bytree=.9,eval_metric='logloss',random_state=SEED,n_jobs=-1)
}

gkf=GroupKFold(n_splits=5)
rows=[]; oof={name:np.full(len(df),np.nan) for name in models}
for fold,(tr,te) in enumerate(gkf.split(X,y,groups),1):
    for name,est in models.items():
        pipe=Pipeline([('pre',clone(pre)),('model',clone(est))])
        pipe.fit(X.iloc[tr],y.iloc[tr]); p=pipe.predict_proba(X.iloc[te])[:,1]
        oof[name][te]=p; pred=(p>=.5).astype(int)
        rows.append({'fold':fold,'model':name,'roc_auc':roc_auc_score(y.iloc[te],p),
                     'pr_auc':average_precision_score(y.iloc[te],p),
                     'balanced_accuracy':balanced_accuracy_score(y.iloc[te],pred),
                     'f1':f1_score(y.iloc[te],pred),'brier':brier_score_loss(y.iloc[te],p),
                     'test_users':groups.iloc[te].nunique(),'test_sessions':len(te)})
fold_metrics=pd.DataFrame(rows)
classification=fold_metrics.groupby('model').agg(
    roc_auc_mean=('roc_auc','mean'),roc_auc_sd=('roc_auc','std'),
    pr_auc_mean=('pr_auc','mean'),balanced_accuracy=('balanced_accuracy','mean'),
    f1=('f1','mean'),brier=('brier','mean')).sort_values('roc_auc_mean',ascending=False).reset_index()
save_table(fold_metrics,'Table_09_Classification_Fold_Metrics')
save_table(classification,'Table_10_Classification_Summary')
best_model=classification.iloc[0]
note('\n4. SESSION CLASSIFICATION (UNSEEN USERS)')
note(f"- Best model: {best_model.model}; ROC-AUC {best_model.roc_auc_mean:.3f} ± {best_model.roc_auc_sd:.3f}")
note('- Outcome-derived user-history features were excluded from this unseen-user benchmark.')
display(classification)

fig,ax=plt.subplots(figsize=(9,4.5))
plot=classification.sort_values('roc_auc_mean')
ax.barh(plot.model,plot.roc_auc_mean,xerr=plot.roc_auc_sd,color='#246A8D',alpha=.85)
ax.axvline(.5,color='k',ls=':'); ax.set(xlabel='User-disjoint ROC-AUC',xlim=(.4,1),title='Figure 4. Leakage-controlled classifier comparison')
fig.tight_layout(); save_figure(fig,'Figure_04_Classifier_Comparison'); plt.show()


### 4.1 Returning-user forecast as a separate operational task

The strict unseen-user task above answers whether a model generalizes to entirely new drivers; its near-chance result is scientifically informative. A charging operator will often face a different task: predicting a **returning user's next session** from information available before that session. The following benchmark is chronological and constructs each user's historical mean with a one-step shift, so no current or future outcome enters the feature.

Results from these two tasks must not be mixed: unseen-user performance measures population transfer, while returning-user performance measures personalized operational prediction.

In [ ]:
# Build a reliable chronological order from the anonymized date field.
def parse_shifted_timestamp_local(series):
    cleaned=series.astype(str).str.replace(r'^(00)(\d{2})-',r'20\2-',regex=True)
    return pd.to_datetime(cleaned,errors='coerce')

date_candidates=[c for c in ['created','Created','startDate','createdDate'] if c in df]
if date_candidates:
    known=df.copy()
    known['timestamp']=parse_shifted_timestamp_local(known[date_candidates[0]])
    known=known.dropna(subset=['timestamp']).sort_values(['timestamp','userId']).copy()
    known['prior_sessions']=known.groupby('userId').cumcount()
    known['user_hist_kwh']=known.groupby('userId').kwhTotal.transform(lambda s:s.expanding().mean().shift())
    known=known.loc[known.prior_sessions.ge(2) & known.user_hist_kwh.notna()].copy()
    known['hour_sin']=np.sin(2*np.pi*known.startTime/24); known['hour_cos']=np.cos(2*np.pi*known.startTime/24)
    known['is_weekend']=known[[c for c in ['Sat','Sun'] if c in known]].sum(axis=1).clip(0,1)
    known_features=['hour_sin','hour_cos','is_weekend','user_hist_kwh','prior_sessions']+[c for c in ['managerVehicle'] if c in known]
    known_cat=[c for c in ['facilityType','platform'] if c in known]
    split=int(.8*len(known)); train=known.iloc[:split]; test=known.iloc[split:]
    # Ensure the evaluation is genuinely on returning users.
    test=test.loc[test.userId.isin(train.userId.unique())]
    yk=(known.kwhTotal>df.kwhTotal.median()).astype(int)
    pre_known=ColumnTransformer([('num',StandardScaler(),known_features),
        ('cat',OneHotEncoder(handle_unknown='ignore',sparse_output=False),known_cat)])
    known_rows=[]
    for name,est in models.items():
        pipe=Pipeline([('pre',clone(pre_known)),('model',clone(est))])
        pipe.fit(train[known_features+known_cat],yk.loc[train.index])
        p=pipe.predict_proba(test[known_features+known_cat])[:,1]; pred=(p>=.5).astype(int)
        known_rows.append({'model':name,'train_sessions':len(train),'test_sessions':len(test),
            'roc_auc':roc_auc_score(yk.loc[test.index],p),'pr_auc':average_precision_score(yk.loc[test.index],p),
            'balanced_accuracy':balanced_accuracy_score(yk.loc[test.index],pred),'f1':f1_score(yk.loc[test.index],pred),
            'brier':brier_score_loss(yk.loc[test.index],p)})
    returning_results=pd.DataFrame(known_rows).sort_values('roc_auc',ascending=False)
    save_table(returning_results,'Table_11_Returning_User_Chronological_Classification')
    note(f"- Returning-user chronological best model: {returning_results.iloc[0].model}; ROC-AUC {returning_results.iloc[0].roc_auc:.3f}")
    note('- Central finding: strong predictability depends on leakage-safe prior user behavior; context alone does not transfer well to unseen users.')
    fig,ax=plt.subplots(figsize=(9,4.5)); q=returning_results.sort_values('roc_auc')
    ax.barh(q.model,q.roc_auc,color='#2B7A78'); ax.axvline(.5,color='k',ls=':')
    ax.set(xlim=(.4,1),xlabel='Chronological ROC-AUC',title='Figure 5. Returning-user next-session prediction')
    fig.tight_layout(); save_figure(fig,'Figure_05_Returning_User_Classification'); plt.show()
else:
    note('- Returning-user benchmark skipped because timestamps could not be parsed.')


In [ ]:
# User-cluster bootstrap for paired OOF AUC differences.
# This respects the main dependence unit better than treating sessions as IID.
top_models=classification.model.head(4).tolist()
best=top_models[0]
comparisons=[]
user_indices={u:np.flatnonzero(groups.values==u) for u in groups.unique()}
for other in top_models[1:]:
    diffs=[]
    for _ in range(N_BOOT):
        sampled=rng.choice(list(user_indices),len(user_indices),replace=True)
        idx=np.concatenate([user_indices[u] for u in sampled])
        if y.iloc[idx].nunique()<2: continue
        diffs.append(roc_auc_score(y.iloc[idx],oof[best][idx])-roc_auc_score(y.iloc[idx],oof[other][idx]))
    comparisons.append({'best_model':best,'comparator':other,'delta_auc':np.mean(diffs),
                        'ci_low':np.quantile(diffs,.025),'ci_high':np.quantile(diffs,.975),
                        'bootstrap_p_two_sided':2*min(np.mean(np.array(diffs)<=0),np.mean(np.array(diffs)>=0))})
comparison_table=pd.DataFrame(comparisons)
save_table(comparison_table,'Table_12_Cluster_Bootstrap_Model_Comparisons')
display(comparison_table)


## 5. Chronological aggregate-load forecasting

Unlike shuffled cross-validation over an average weekday-hour surface, this section reconstructs a chronological series from the anonymized timestamp fields and uses rolling-origin validation. The anonymized year is replaced only to make the sequence parseable; calendar ordering is preserved.

> **Modeling convention:** each session's constant average power is distributed across all overlapping clock hours. This preserves session energy and is more physical than assigning the full session to its start hour, although it remains a reconstruction rather than interval-metered feeder demand.

In [ ]:
def parse_shifted_timestamp(series):
    # Convert years such as 0014 to 2014 while preserving month/day/time.
    cleaned=series.astype(str).str.replace(r'^(00)(\d{2})-',r'20\2-',regex=True)
    return pd.to_datetime(cleaned,errors='coerce')

date_candidates=[c for c in ['created','Created','startDate','createdDate'] if c in df]
if date_candidates:
    ts=parse_shifted_timestamp(df[date_candidates[0]])
else:
    ts=pd.Series(pd.NaT,index=df.index)

if ts.notna().mean()<.8:
    note('- WARNING: a reliable chronological timestamp could not be parsed; forecast section skipped.')
    forecast_summary=pd.DataFrame([{'status':'skipped','reason':'timestamp unavailable'}])
    save_table(forecast_summary,'Table_13_Chronological_Forecast')
else:
    load=df.assign(timestamp=ts,avg_kw=df.kwhTotal/df.chargeTimeHrs).dropna(subset=['timestamp'])
    # Reconstruct a physically more meaningful hourly profile. Constant session-average
    # power is allocated to every overlapping clock hour, weighted by the overlap.
    # This preserves session energy and is preferable to assigning all power to the start hour.
    hourly_energy={}
    for row in load[['timestamp','chargeTimeHrs','avg_kw']].itertuples(index=False):
        start=row.timestamp; end=start+pd.to_timedelta(row.chargeTimeHrs,unit='h')
        hour=start.floor('h')
        while hour<end:
            overlap=max(0,(min(end,hour+pd.Timedelta(hours=1))-max(start,hour)).total_seconds()/3600)
            hourly_energy[hour]=hourly_energy.get(hour,0.)+row.avg_kw*overlap
            hour+=pd.Timedelta(hours=1)
    hourly=pd.Series(hourly_energy,name='load_kw').sort_index().asfreq('h',fill_value=0).to_frame()
    hourly['hour_sin']=np.sin(2*np.pi*hourly.index.hour/24)
    hourly['hour_cos']=np.cos(2*np.pi*hourly.index.hour/24)
    hourly['dow_sin']=np.sin(2*np.pi*hourly.index.dayofweek/7)
    hourly['dow_cos']=np.cos(2*np.pi*hourly.index.dayofweek/7)
    for lag in [1,24,168]: hourly[f'lag_{lag}']=hourly.load_kw.shift(lag)
    hourly['roll_24']=hourly.load_kw.shift(1).rolling(24).mean()
    hourly=hourly.dropna()
    fcols=[c for c in hourly if c!='load_kw']
    splits=TimeSeriesSplit(n_splits=5)
    pred=np.full(len(hourly),np.nan); naive=np.full(len(hourly),np.nan); fold_id=np.full(len(hourly),np.nan)
    lower=np.full(len(hourly),np.nan); upper=np.full(len(hourly),np.nan)
    from xgboost import XGBRegressor
    for fold,(tr,te) in enumerate(splits.split(hourly),1):
        model=XGBRegressor(n_estimators=400,max_depth=4,learning_rate=.04,subsample=.85,
                           colsample_bytree=.9,objective='reg:squarederror',random_state=SEED,n_jobs=-1)
        # Split the chronological training window into proper training and conformal
        # calibration portions. The calibration residual quantile yields a finite-sample,
        # distribution-free 90% prediction interval under exchangeability.
        cut=max(1,int(.8*len(tr))); fit_idx=tr[:cut]; cal_idx=tr[cut:]
        model.fit(hourly.iloc[fit_idx][fcols],hourly.iloc[fit_idx].load_kw)
        cal_pred=model.predict(hourly.iloc[cal_idx][fcols])
        q=np.quantile(np.abs(hourly.iloc[cal_idx].load_kw.values-cal_pred),.90,method='higher')
        pred[te]=model.predict(hourly.iloc[te][fcols]); naive[te]=hourly.iloc[te].lag_168; fold_id[te]=fold
        lower[te]=np.maximum(0,pred[te]-q); upper[te]=pred[te]+q
    valid=~np.isnan(pred); actual=hourly.load_kw.values
    def regrow(name,p):
        return {'model':name,'MAE_kW':mean_absolute_error(actual[valid],p[valid]),
                'RMSE_kW':mean_squared_error(actual[valid],p[valid])**.5,
                'nRMSE_pct':100*mean_squared_error(actual[valid],p[valid])**.5/(actual[valid].mean()+1e-9),
                'R2':r2_score(actual[valid],p[valid])}
    forecast_summary=pd.DataFrame([regrow('XGBoost rolling-origin',pred),regrow('Seasonal naive (168 h)',naive)])
    forecast_summary['PI90_coverage']=np.nan; forecast_summary['PI90_mean_width_kW']=np.nan
    forecast_summary.loc[0,'PI90_coverage']=np.mean((actual[valid]>=lower[valid])&(actual[valid]<=upper[valid]))
    forecast_summary.loc[0,'PI90_mean_width_kW']=np.mean(upper[valid]-lower[valid])
    save_table(forecast_summary,'Table_13_Chronological_Forecast')
    forecast_predictions=pd.DataFrame({'time':hourly.index,'actual_kw':actual,'forecast_kw':pred,
        'lower_90':lower,'upper_90':upper,'seasonal_naive_kw':naive,'fold':fold_id})
    save_table(forecast_predictions.loc[valid],'Table_14_Forecast_Predictions')
    diag=forecast_predictions.loc[valid].copy(); diag['month']=pd.to_datetime(diag.time).dt.month
    diag['hour']=pd.to_datetime(diag.time).dt.hour; diag['peak_period']=diag.hour.between(8,18)
    monthly=diag.groupby('month').apply(lambda z:pd.Series({'n':len(z),'MAE_kW':mean_absolute_error(z.actual_kw,z.forecast_kw),
        'RMSE_kW':mean_squared_error(z.actual_kw,z.forecast_kw)**.5,'PI90_coverage':np.mean((z.actual_kw>=z.lower_90)&(z.actual_kw<=z.upper_90))})).reset_index()
    peak=diag.groupby('peak_period').apply(lambda z:pd.Series({'n':len(z),'MAE_kW':mean_absolute_error(z.actual_kw,z.forecast_kw),
        'RMSE_kW':mean_squared_error(z.actual_kw,z.forecast_kw)**.5,'PI90_coverage':np.mean((z.actual_kw>=z.lower_90)&(z.actual_kw<=z.upper_90))})).reset_index()
    save_table(monthly,'Table_15_Forecast_By_Month'); save_table(peak,'Table_16_Forecast_Peak_Offpeak')
    note('\n5. CHRONOLOGICAL LOAD FORECAST')
    note(f"- XGBoost rolling-origin RMSE: {forecast_summary.iloc[0].RMSE_kW:.2f} kW; R²: {forecast_summary.iloc[0].R2:.3f}")
    note(f"- Seasonal-naive RMSE: {forecast_summary.iloc[1].RMSE_kW:.2f} kW; R²: {forecast_summary.iloc[1].R2:.3f}")
    note(f"- Conformal 90% interval coverage: {forecast_summary.iloc[0].PI90_coverage:.3f}; mean width {forecast_summary.iloc[0].PI90_mean_width_kW:.2f} kW")
    note('- Load is an energy-preserving hourly reconstruction, not interval-metered feeder demand.')
    fig,axes=plt.subplots(2,1,figsize=(14,7),sharex=False)
    view=forecast_predictions.loc[valid].tail(24*21)
    axes[0].plot(view.time,view.actual_kw,label='Observed proxy',lw=1)
    axes[0].plot(view.time,view.forecast_kw,label='XGBoost',lw=1.2)
    axes[0].fill_between(view.time,view.lower_90,view.upper_90,color='#246A8D',alpha=.15,label='90% conformal interval')
    axes[0].plot(view.time,view.seasonal_naive_kw,label='Seasonal naive',alpha=.7)
    axes[0].set(ylabel='kW',title='Last 21 validation days'); axes[0].legend(ncol=3)
    axes[1].scatter(pred[valid],actual[valid]-pred[valid],s=8,alpha=.35,color='#2B7A78')
    axes[1].axhline(0,color='k',ls=':'); axes[1].set(xlabel='Predicted kW',ylabel='Residual (kW)',title='Rolling-origin residuals')
    fig.suptitle('Figure 6. Chronological load-forecast validation',fontweight='bold')
    fig.tight_layout(); save_figure(fig,'Figure_06_Chronological_Forecast'); plt.show()


## 6. Scenario-based dynamic pricing with stronger baselines

This is a **conditional simulation**, not an empirical estimate of realized revenue. The environment includes deferred flexible demand (backlog), a service penalty, and a capacity penalty, which create intertemporal coupling. Policy comparisons include:

- a flat reference price;
- exhaustive best constant price;
- a transparent time-of-use rule;
- tabular Q-learning.

Performance is evaluated across multiple random seeds and across elasticity assumptions. Physical feeder power flow is still outside this dataset and must be added for a fully IJEPES-aligned system study.

In [ ]:
# Use the empirical hourly shape if it exists; otherwise derive a weekday-hour profile.
if 'hourly' in globals() and len(hourly):
    base_shape=hourly.groupby(hourly.index.hour).load_kw.mean().reindex(range(24),fill_value=0).values
else:
    tmp=df.assign(avg_kw=df.kwhTotal/df.chargeTimeHrs)
    base_shape=tmp.groupby('startTime').avg_kw.sum().reindex(range(24),fill_value=0).values
base_shape=base_shape/(base_shape.max()+1e-9)

class PricingEnv:
    """Small stochastic environment with backlog and a capacity limit."""
    def __init__(self,elasticity=1.0,arrival_scale=1.0,capacity=.85,seed=0):
        self.elasticity=elasticity; self.arrival_scale=arrival_scale; self.capacity=capacity
        self.prices=np.array([.10,.20,.30,.40,.50]); self.ref=.25
        self.rng=np.random.default_rng(seed)
    def reset(self):
        self.t=0; self.backlog=0.; self.total_revenue=0.; self.loads=[]; self.service=[]; return self.state()
    def state(self):
        return np.array([self.t/23,base_shape[self.t],min(self.backlog,2.)/2])
    def step(self,action):
        price=self.prices[action]
        arrivals=max(0,base_shape[self.t]*self.arrival_scale+self.rng.normal(0,.04))
        essential=.30*arrivals
        flexible=.70*arrivals+self.backlog
        willingness=np.clip(np.exp(-self.elasticity*(price-self.ref)/self.ref),.05,1.)
        desired=essential+flexible*willingness
        served=min(desired,self.capacity)
        self.backlog=max(0,arrivals+self.backlog-served)
        revenue=price*served
        overload=max(0,desired-self.capacity)
        reward=revenue-0.35*self.backlog-1.5*overload**2
        self.total_revenue+=revenue; self.loads.append(served); self.service.append(arrivals)
        self.t+=1; done=self.t==24
        return (np.zeros(3) if done else self.state()),reward,done
    def metrics(self):
        demand=sum(self.service); served=sum(self.loads)
        return {'revenue':self.total_revenue,'peak_load':max(self.loads),'load_variance':np.var(self.loads),
                'service_ratio':min(served/(demand+1e-9),1.),'final_backlog':self.backlog}

def evaluate_rule(rule,elasticity,seed):
    env=PricingEnv(elasticity=elasticity,seed=seed); state=env.reset(); total=0; done=False
    while not done:
        action=rule(env,state); state,r,done=env.step(action); total+=r
    return {'reward':total,**env.metrics()}

flat=lambda env,s:2
tou=lambda env,s:3 if 8<=env.t<=18 else 1

def best_constant(elasticity,seed):
    candidates=[]
    for a in range(5):
        out=evaluate_rule(lambda env,s,a=a:a,elasticity,seed); candidates.append((out['reward'],a,out))
    return max(candidates,key=lambda z:z[0])

def discretize(state):
    return (min(int(state[0]*6),5),min(int(state[1]*4),3),min(int(state[2]*4),3))

def train_tabular_q(elasticity,seed,episodes=500):
    rng=np.random.default_rng(seed); Q=np.zeros((6,4,4,5)); rewards=[]
    for ep in range(episodes):
        env=PricingEnv(elasticity=elasticity,seed=seed*10000+ep); s=env.reset(); total=0; done=False
        eps=max(.05,1-ep/episodes)
        while not done:
            ds=discretize(s); a=rng.integers(5) if rng.random()<eps else int(np.argmax(Q[ds]))
            s2,r,done=env.step(a); target=r if done else r+.98*np.max(Q[discretize(s2)])
            Q[ds+(a,)]+=.12*(target-Q[ds+(a,)]); s=s2; total+=r
        rewards.append(total)
    rule=lambda env,s:int(np.argmax(Q[discretize(s)]))
    out=evaluate_rule(rule,elasticity,seed+900000)
    return out,rewards,Q

elasticities=[.5,1.,2.,3.]
policy_rows=[]; learning_curves={}
for elasticity in elasticities:
    for seed in range(N_AGENT_SEEDS):
        for name,rule in [('Flat 0.30',flat),('TOU rule',tou)]:
            policy_rows.append({'elasticity':elasticity,'seed':seed,'policy':name,**evaluate_rule(rule,elasticity,seed)})
        _,a,bestout=best_constant(elasticity,seed)
        policy_rows.append({'elasticity':elasticity,'seed':seed,'policy':'Best constant',
                            'action':a,'selected_price':PricingEnv(elasticity=elasticity).prices[a],**bestout})
        qout,curve,Q=train_tabular_q(elasticity,seed,episodes=150 if FAST_MODE else 600)
        policy_rows.append({'elasticity':elasticity,'seed':seed,'policy':'Tabular Q-learning',**qout})
        if seed==0: learning_curves[elasticity]=curve
policy_results=pd.DataFrame(policy_rows)
policy_summary=policy_results.groupby(['elasticity','policy']).agg(
    reward_mean=('reward','mean'),reward_sd=('reward','std'),revenue_mean=('revenue','mean'),
    peak_load=('peak_load','mean'),load_variance=('load_variance','mean'),
    service_ratio=('service_ratio','mean'),final_backlog=('final_backlog','mean'),
    selected_price=('selected_price','mean')).reset_index()
save_table(policy_results,'Table_17_Policy_Seed_Results')
save_table(policy_summary,'Table_18_Policy_Sensitivity_Summary')
note('\n6. SCENARIO-BASED PRICING CONTROL')
note(f'- Policies evaluated across {len(elasticities)} elasticity assumptions and {N_AGENT_SEEDS} seeds.')
note('- Results are conditional simulation outcomes, not empirically identified revenue effects.')
display(policy_summary)

fig,axes=plt.subplots(1,2,figsize=(13,4.5))
sns.lineplot(data=policy_summary,x='elasticity',y='reward_mean',hue='policy',marker='o',ax=axes[0])
axes[0].set(title='Policy reward across assumed elasticities',ylabel='Mean simulated reward')
sns.scatterplot(data=policy_summary,x='revenue_mean',y='reward_mean',hue='policy',style='elasticity',s=100,ax=axes[1])
axes[1].set(title='Revenue versus constrained objective',xlabel='Mean simulated revenue',ylabel='Mean reward')
fig.suptitle('Figure 7. Sensitivity-aware policy comparison',fontweight='bold')
fig.tight_layout(); save_figure(fig,'Figure_07_Policy_Sensitivity'); plt.show()


## 7. Publication-grade quantum-classical benchmarks

The VQC is evaluated on all five user-disjoint folds and multiple seeds. Every quantum run is paired with compact classical models trained on the **identical capped train/test subsets**. The notebook reports fold/seed dispersion, confidence intervals, parameter count, and runtime. A representative trained circuit is additionally evaluated with finite shots and depolarizing noise.

These experiments remain simulator studies. They test competitiveness and robustness; they do not establish quantum advantage or hardware readiness.

In [ ]:
quantum_results=[]; quantum_robustness=[]; representative=None
if RUN_QUANTUM:
    import pennylane as qml
    from pennylane import numpy as pnp
    from sklearn.decomposition import PCA
    from sklearn.neural_network import MLPClassifier

    nq=4; layers=2
    quantum_start=time.time(); completed_vqc_runs=0
    total_vqc_runs=gkf.get_n_splits()*N_QUANTUM_SEEDS
    print(f'Quantum benchmark started: {total_vqc_runs} VQC runs '
          f'({gkf.get_n_splits()} folds x {N_QUANTUM_SEEDS} seeds).',flush=True)
    print(f'Each run uses up to {QUANTUM_CAP_PER_SPLIT} balanced training and test cases '
          f'for {QUANTUM_STEPS} optimization steps.',flush=True)

    def progress_message(label,done,total,start):
        elapsed=time.time()-start
        rate=elapsed/max(done,1)
        remaining=rate*max(total-done,0)
        print(f'[{done}/{total}] {label} | elapsed {elapsed/60:.1f} min | '
              f'estimated remaining {remaining/60:.1f} min',flush=True)
    def balanced_cap(yv,cap,rng):
        parts=[]
        for cls in [0,1]:
            ids=np.flatnonzero(np.asarray(yv)==cls); take=min(len(ids),cap//2)
            parts.append(rng.choice(ids,take,replace=False))
        return np.concatenate(parts)

    for fold,(tr,te) in enumerate(gkf.split(X,y,groups),1):
        print(f'\nPreparing user-disjoint fold {fold}/5 ...',flush=True)
        prep=clone(pre).fit(X.iloc[tr]); A=prep.transform(X.iloc[tr]); B=prep.transform(X.iloc[te])
        pca=PCA(nq,random_state=SEED).fit(A)
        A=np.tanh(pca.transform(A))*np.pi; B=np.tanh(pca.transform(B))*np.pi
        for qseed in range(N_QUANTUM_SEEDS):
            run_number=completed_vqc_runs+1
            print(f'  Starting fold {fold}/5, quantum seed {qseed+1}/{N_QUANTUM_SEEDS} '
                  f'(VQC run {run_number}/{total_vqc_runs}) ...',flush=True)
            rngq=np.random.default_rng(SEED+100*fold+qseed)
            itr=balanced_cap(y.iloc[tr].values,QUANTUM_CAP_PER_SPLIT,rngq)
            ite=balanced_cap(y.iloc[te].values,QUANTUM_CAP_PER_SPLIT,rngq)
            Aq,Bq=A[itr],B[ite]; ytr,yte=y.iloc[tr].values[itr],y.iloc[te].values[ite]

            # Matched classical baselines on exactly the same PCA features and cases.
            classical=[('Matched logistic',LogisticRegression(max_iter=1500)),
                       ('Matched compact MLP',MLPClassifier((3,),max_iter=1000,random_state=qseed))]
            for name,model in classical:
                st=time.time(); model.fit(Aq,ytr); score=model.predict_proba(Bq)[:,1]
                npar=(nq+1)*2 if name=='Matched logistic' else (nq*3+3)+(3+1)
                quantum_results.append({'model':name,'fold':fold,'seed':qseed,'n_train':len(Aq),'n_test':len(Bq),
                    'qubits':0,'layers':0,'trainable_parameters':npar,'steps':np.nan,
                    'roc_auc':roc_auc_score(yte,score),'runtime_seconds':time.time()-st,'evaluation':'analytic'})
                print(f'    {name} complete: ROC-AUC={roc_auc_score(yte,score):.3f}',flush=True)

            dev=qml.device('lightning.qubit',wires=nq)
            @qml.qnode(dev,diff_method='adjoint')
            def circuit(w,x):
                qml.AngleEmbedding(x,wires=range(nq),rotation='Y')
                qml.StronglyEntanglingLayers(w,wires=range(nq))
                return qml.expval(qml.PauliZ(0))
            w=pnp.array(rngq.normal(0,.1,qml.StronglyEntanglingLayers.shape(layers,nq)),requires_grad=True)
            bias=pnp.array(0.,requires_grad=True); Xp=pnp.array(Aq,requires_grad=False)
            target=pnp.array(2*ytr-1,requires_grad=False); opt=qml.NesterovMomentumOptimizer(.08)
            trajectory=[]; st=time.time()
            def loss(w,bias): return pnp.mean((pnp.stack([circuit(w,x) for x in Xp])+bias-target)**2)
            for step in range(QUANTUM_STEPS):
                (w,bias),value=opt.step_and_cost(loss,w,bias); trajectory.append(float(value))
                report_every=max(1,QUANTUM_STEPS//5)
                if step==0 or (step+1)%report_every==0 or step+1==QUANTUM_STEPS:
                    print(f'    VQC optimization step {step+1}/{QUANTUM_STEPS}; loss={float(value):.6f}',flush=True)
            score=np.array([float(circuit(w,x))+float(bias) for x in Bq])
            quantum_results.append({'model':'VQC','fold':fold,'seed':qseed,'n_train':len(Aq),'n_test':len(Bq),
                'qubits':nq,'layers':layers,'trainable_parameters':int(np.prod(w.shape))+1,'steps':QUANTUM_STEPS,
                'roc_auc':roc_auc_score(yte,score),'runtime_seconds':time.time()-st,'evaluation':'analytic'})
            if representative is None: representative=(w.copy(),float(bias),Bq.copy(),yte.copy(),trajectory)
            completed_vqc_runs+=1
            # Lightweight checkpoint protects completed fold/seed results if Colab disconnects.
            pd.DataFrame(quantum_results).to_csv(TABLE_DIR/'Table_19_Quantum_Matched_Run_Results_checkpoint.csv',index=False)
            progress_message(f'fold {fold}, seed {qseed+1} complete; VQC ROC-AUC={roc_auc_score(yte,score):.3f}',
                             completed_vqc_runs,total_vqc_runs,quantum_start)

    qtable=pd.DataFrame(quantum_results)
    qsummary=qtable.groupby('model').agg(runs=('roc_auc','size'),roc_auc_mean=('roc_auc','mean'),
        roc_auc_sd=('roc_auc','std'),runtime_mean_s=('runtime_seconds','mean'),
        parameters=('trainable_parameters','median')).reset_index()
    qsummary['ci95_low']=qsummary.roc_auc_mean-1.96*qsummary.roc_auc_sd/np.sqrt(qsummary.runs)
    qsummary['ci95_high']=qsummary.roc_auc_mean+1.96*qsummary.roc_auc_sd/np.sqrt(qsummary.runs)
    save_table(qtable,'Table_19_Quantum_Matched_Run_Results'); save_table(qsummary,'Table_20_Quantum_Matched_Summary')
    paired=[]
    qp=qtable.pivot_table(index=['fold','seed'],columns='model',values='roc_auc')
    for comparator in ['Matched logistic','Matched compact MLP']:
        d=(qp['VQC']-qp[comparator]).dropna(); se=d.std(ddof=1)/np.sqrt(len(d))
        tcrit=stats.t.ppf(.975,len(d)-1)
        try: wp=stats.wilcoxon(d).pvalue
        except ValueError: wp=np.nan
        paired.append({'comparison':f'VQC - {comparator}','pairs':len(d),'mean_delta_auc':d.mean(),
            'ci95_low':d.mean()-tcrit*se,'ci95_high':d.mean()+tcrit*se,
            'paired_t_p':stats.ttest_1samp(d,0).pvalue,'wilcoxon_p':wp})
    paired_quantum=pd.DataFrame(paired); save_table(paired_quantum,'Table_20A_Quantum_Paired_Comparisons')

    # Representative finite-shot and noise evaluation (no retraining).
    w0,b0,B0,y0,traj=representative
    for shots in [128,512,2048]:
        print(f'Evaluating representative circuit with {shots} shots ...',flush=True)
        shotdev=qml.device('default.qubit',wires=nq,shots=shots,seed=SEED)
        @qml.qnode(shotdev)
        def shot_circuit(x):
            qml.AngleEmbedding(x,wires=range(nq),rotation='Y'); qml.StronglyEntanglingLayers(w0,wires=range(nq))
            return qml.expval(qml.PauliZ(0))
        sc=np.array([float(shot_circuit(x))+b0 for x in B0])
        quantum_robustness.append({'condition':f'{shots} shots','roc_auc':roc_auc_score(y0,sc)})
        print(f'  {shots}-shot ROC-AUC={roc_auc_score(y0,sc):.3f}',flush=True)
    for noise in [.001,.005,.01,.02]:
        print(f'Evaluating depolarizing noise p={noise} ...',flush=True)
        noised=qml.device('default.mixed',wires=nq)
        @qml.qnode(noised)
        def noisy_circuit(x):
            qml.AngleEmbedding(x,wires=range(nq),rotation='Y')
            for wire in range(nq): qml.DepolarizingChannel(noise,wires=wire)
            qml.StronglyEntanglingLayers(w0,wires=range(nq)); return qml.expval(qml.PauliZ(0))
        sc=np.array([float(noisy_circuit(x))+b0 for x in B0])
        quantum_robustness.append({'condition':f'depolarizing p={noise}','roc_auc':roc_auc_score(y0,sc)})
        print(f'  noise p={noise} ROC-AUC={roc_auc_score(y0,sc):.3f}',flush=True)
    quantum_robustness=pd.DataFrame(quantum_robustness); save_table(quantum_robustness,'Table_21_Quantum_Shot_Noise_Robustness')
    note('\n7. QUANTUM-CLASSICAL CLASSIFICATION')
    for row in qsummary.itertuples(): note(f'- {row.model}: ROC-AUC {row.roc_auc_mean:.3f} ± {row.roc_auc_sd:.3f} ({row.runs} fold-seed runs)')
    note('- Quantum results are exploratory simulator evidence; no quantum advantage is claimed.')
    print(f'Quantum classification benchmark finished in {(time.time()-quantum_start)/60:.1f} minutes.',flush=True)
    fig,axes=plt.subplots(1,2,figsize=(12,4.2))
    sns.boxplot(data=qtable,x='model',y='roc_auc',ax=axes[0]); axes[0].axhline(.5,color='k',ls=':'); axes[0].set_title('Matched fold-seed ROC-AUC')
    axes[1].plot(traj,color='#2B7A78'); axes[1].set(xlabel='Optimization step',ylabel='Training loss',title='Representative VQC convergence')
    fig.suptitle('Figure 8. Matched quantum-classical classification',fontweight='bold')
    fig.tight_layout(); save_figure(fig,'Figure_08_Quantum_Matched_Benchmark'); plt.show()
else:
    note('\n7. QUANTUM-CLASSICAL CLASSIFICATION')
    note('- Skipped because RUN_QUANTUM=False.')


### 7.1 Restored Q-EVCS pricing experiment

The manuscript's principal quantum-control claim is reproduced with a five-qubit variational Q-function. Q-EVCS and a compact tabular Q-learning comparator receive the same environment, elasticity assumption, episode budget, evaluation seeds, and reward definition. Multi-seed results—not the best seed—are emphasized.

In [ ]:
qevcs_results=[]
if RUN_QUANTUM:
    def train_qevcs(seed,elasticity=1.0,episodes=None,nq=5,layers=2):
        episodes=episodes or (25 if FAST_MODE else 60)
        rng=np.random.default_rng(SEED+seed); dev=qml.device('lightning.qubit',wires=nq)
        @qml.qnode(dev,diff_method='adjoint')
        def qcircuit(w,x):
            qml.AngleEmbedding(x,wires=range(nq),rotation='Y')
            qml.StronglyEntanglingLayers(w,wires=range(nq))
            return [qml.expval(qml.PauliZ(i)) for i in range(nq)]
        w=pnp.array(rng.normal(0,.1,qml.StronglyEntanglingLayers.shape(layers,nq)),requires_grad=True)
        scale=pnp.array(np.ones(5),requires_grad=True); bias=pnp.array(np.zeros(5),requires_grad=True)
        opt=qml.AdamOptimizer(.05); buffer=[]; curve=[]
        enc=lambda s:np.array([s[0]*np.pi,s[1]*np.pi,s[2]*np.pi,np.sin(s[0]*np.pi),np.cos(s[0]*np.pi)])
        def qvalue(w,scale,bias,x): return pnp.stack(qcircuit(w,x))*scale+bias
        st=time.time()
        for ep in range(episodes):
            env=PricingEnv(elasticity=elasticity,seed=seed*10000+ep); s=env.reset(); total=0; done=False
            eps=max(.05,1-ep/episodes)
            while not done:
                a=rng.integers(5) if rng.random()<eps else int(np.argmax(np.array(qvalue(w,scale,bias,pnp.array(enc(s),requires_grad=False)))))
                s2,r,done=env.step(a); buffer.append((enc(s),a,r,enc(s2),done)); s=s2; total+=r
                if len(buffer)>600: buffer.pop(0)
            if len(buffer)>=16:
                for _ in range(2):
                    batch=[buffer[i] for i in rng.integers(0,len(buffer),12)]
                    targets=[]
                    for x,a,r,x2,d in batch:
                        nxt=0 if d else .98*float(np.max(np.array(qvalue(w,scale,bias,pnp.array(x2,requires_grad=False)))))
                        targets.append(r+nxt)
                    def loss(w,scale,bias):
                        return pnp.mean(pnp.stack([(qvalue(w,scale,bias,pnp.array(x,requires_grad=False))[a]-targets[j])**2
                                                   for j,(x,a,r,x2,d) in enumerate(batch)]))
                    w,scale,bias=opt.step(loss,w,scale,bias)
            curve.append(total)
        rule=lambda env,s:int(np.argmax(np.array(qvalue(w,scale,bias,pnp.array(enc(s),requires_grad=False)))))
        evaluations=[]
        for evseed in range(10): evaluations.append(evaluate_rule(rule,elasticity,900000+evseed))
        metrics=pd.DataFrame(evaluations).mean(numeric_only=True).to_dict()
        return {'policy':'Q-EVCS','seed':seed,'episodes':episodes,'parameters':int(np.prod(w.shape))+10,
                'runtime_seconds':time.time()-st,**metrics},curve

    qcurves=[]; episodes=25 if FAST_MODE else 60
    for seed in range(N_QUANTUM_SEEDS):
        qrow,curve=train_qevcs(seed,episodes=episodes); qevcs_results.append(qrow); qcurves.append(curve)
        cout,ccurve,_=train_tabular_q(1.0,seed,episodes=episodes)
        qevcs_results.append({'policy':'Matched tabular Q-learning','seed':seed,'episodes':episodes,'parameters':6*4*4*5,
                              'runtime_seconds':np.nan,**cout})
    qevcs_results=pd.DataFrame(qevcs_results)
    qevcs_summary=qevcs_results.groupby('policy').agg(seeds=('seed','nunique'),reward_mean=('reward','mean'),
        reward_sd=('reward','std'),revenue_mean=('revenue','mean'),peak_load=('peak_load','mean'),
        service_ratio=('service_ratio','mean'),runtime_mean_s=('runtime_seconds','mean'),parameters=('parameters','median')).reset_index()
    save_table(qevcs_results,'Table_22_QEVCS_Seed_Results'); save_table(qevcs_summary,'Table_23_QEVCS_Summary')
    note('\n7.1 Q-EVCS PRICING')
    for row in qevcs_summary.itertuples(): note(f'- {row.policy}: reward {row.reward_mean:.3f} ± {row.reward_sd:.3f} across {row.seeds} seeds')
    note('- Q-EVCS is an exploratory simulated controller; no quantum advantage is claimed.')
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for i,c in enumerate(qcurves): axes[0].plot(pd.Series(c).rolling(8,min_periods=1).mean(),alpha=.65,label=f'seed {i}')
    axes[0].set(xlabel='Episode',ylabel='Rolling reward',title='Q-EVCS learning across seeds'); axes[0].legend(ncol=2,fontsize=8)
    sns.boxplot(data=qevcs_results,x='policy',y='reward',ax=axes[1]); axes[1].set_title('Matched-budget evaluation')
    fig.suptitle('Figure 9. Q-EVCS multi-seed pricing benchmark',fontweight='bold')
    fig.tight_layout(); save_figure(fig,'Figure_09_QEVCS_Multiseed'); plt.show()
else:
    note('\n7.1 Q-EVCS PRICING\n- Skipped because RUN_QUANTUM=False.')


## 8. Distribution-feeder interaction and forecast-error propagation

A standard IEEE 33-bus distribution feeder is used to translate workplace charging policies into power-system metrics. The EV facility is connected at bus 17. The non-EV benchmark demand is scaled to 55% to create a feasible reference state, explicit 0.40-kA line ratings are imposed, and a synthetic 3-MVA point-of-common-coupling limit is used. Hourly AC power flows report minimum voltage, voltage violations, maximum line loading, feeder losses, and overloads. Forecast uncertainty is propagated by sampling empirical rolling-origin forecast residuals.

This is a benchmark-system study, not a claim about the unidentified physical feeder serving the original workplace.

In [ ]:
import pandapower as pp
import pandapower.networks as pn

def policy_load_profile(rule,elasticity=1.0,seed=0):
    env=PricingEnv(elasticity=elasticity,seed=seed); s=env.reset(); done=False
    while not done:
        a=rule(env,s); s,_,done=env.step(a)
    return np.asarray(env.loads)

# Representative policies. The learned tabular policy is regenerated deterministically.
_,best_action,_=best_constant(1.0,SEED)
_,_,Qgrid=train_tabular_q(1.0,SEED,episodes=150 if FAST_MODE else 600)
qrule=lambda env,s:int(np.argmax(Qgrid[discretize(s)]))
network_policies={'Flat 0.30':flat,'TOU rule':tou,'Best constant':lambda env,s:best_action,
                  'Tabular Q-learning':qrule}

def feeder_metrics(profile,error_factor=None,ev_peak_mw=.60):
    net=pn.case33bw()
    # case33bw is deliberately stressed and uses placeholder ampacities. A feasible
    # non-EV reference and explicit ratings make incremental EV impacts interpretable.
    net.load.loc[:,['p_mw','q_mvar']]*=.55
    net.line.loc[:,'max_i_ka']=.40
    ev_idx=pp.create_load(net,bus=17,p_mw=0,q_mvar=0,name='Workplace EV aggregate')
    rows=[]; error_factor=np.ones(24) if error_factor is None else np.asarray(error_factor)
    for hour,value in enumerate(profile):
        net.load.at[ev_idx,'p_mw']=max(0,ev_peak_mw*value*error_factor[hour]); net.load.at[ev_idx,'q_mvar']=0
        try:
            pp.runpp(net,algorithm='nr',init='auto',max_iteration=30)
            pcc_p=float(abs(net.res_ext_grid.p_mw.iloc[0])); pcc_q=float(abs(net.res_ext_grid.q_mvar.iloc[0]))
            pcc_mva=(pcc_p**2+pcc_q**2)**.5
            rows.append({'hour':hour,'min_voltage_pu':net.res_bus.vm_pu.min(),
                'voltage_violations':int(((net.res_bus.vm_pu<.95)|(net.res_bus.vm_pu>1.05)).sum()),
                'max_line_loading_pct':net.res_line.loading_percent.max(),
                'line_overloads':int((net.res_line.loading_percent>100).sum()),
                'losses_mw':net.res_line.pl_mw.sum(),'pcc_mva':pcc_mva,'transformer_overload':int(pcc_mva>3.0)})
        except pp.LoadflowNotConverged:
            rows.append({'hour':hour,'min_voltage_pu':np.nan,'voltage_violations':len(net.bus),
                'max_line_loading_pct':np.nan,'line_overloads':len(net.line),'losses_mw':np.nan,
                'pcc_mva':np.nan,'transformer_overload':1})
    return pd.DataFrame(rows)

baseline_grid=feeder_metrics(np.zeros(24)); baseline_grid['policy']='No-EV reference'; baseline_grid['scenario']=0
network_hourly=[baseline_grid]
network_summary=[{'policy':'No-EV reference','scenarios':1,
    'min_voltage_mean':baseline_grid.min_voltage_pu.mean(),'min_voltage_worst':baseline_grid.min_voltage_pu.min(),
    'voltage_violation_hours_mean':int((baseline_grid.voltage_violations>0).sum()),
    'max_line_loading_mean':baseline_grid.max_line_loading_pct.mean(),'max_line_loading_worst':baseline_grid.max_line_loading_pct.max(),
    'line_overload_hours_mean':int((baseline_grid.line_overloads>0).sum()),'losses_mwh_mean':baseline_grid.losses_mw.sum(),
    'transformer_overload_hours_mean':int(baseline_grid.transformer_overload.sum())}]
residual_pool=(forecast_predictions.loc[valid].actual_kw-forecast_predictions.loc[valid].forecast_kw).values if 'forecast_predictions' in globals() else np.array([0.])
scale=max(float(forecast_predictions.loc[valid].actual_kw.mean()),1e-6) if 'forecast_predictions' in globals() else 1
N_GRID_SCENARIOS=20 if FAST_MODE else 100
for name,rule in network_policies.items():
    profile=policy_load_profile(rule,1.0,SEED)
    central=feeder_metrics(profile); central['policy']=name; central['scenario']=0; network_hourly.append(central)
    scenario_metrics=[]
    for scenario in range(N_GRID_SCENARIOS):
        factor=np.clip(1+rng.choice(residual_pool,24,replace=True)/scale,.25,2.0)
        rr=feeder_metrics(profile,factor)
        scenario_metrics.append({'policy':name,'scenario':scenario+1,'min_voltage_pu':rr.min_voltage_pu.min(),
            'voltage_violation_hours':int((rr.voltage_violations>0).sum()),
            'max_line_loading_pct':rr.max_line_loading_pct.max(),'line_overload_hours':int((rr.line_overloads>0).sum()),
            'losses_mwh':rr.losses_mw.sum(),'transformer_overload_hours':int(rr.transformer_overload.sum())})
    ss=pd.DataFrame(scenario_metrics)
    network_summary.append({'policy':name,'scenarios':N_GRID_SCENARIOS,
        'min_voltage_mean':ss.min_voltage_pu.mean(),'min_voltage_worst':ss.min_voltage_pu.min(),
        'voltage_violation_hours_mean':ss.voltage_violation_hours.mean(),
        'max_line_loading_mean':ss.max_line_loading_pct.mean(),'max_line_loading_worst':ss.max_line_loading_pct.max(),
        'line_overload_hours_mean':ss.line_overload_hours.mean(),'losses_mwh_mean':ss.losses_mwh.mean(),
        'transformer_overload_hours_mean':ss.transformer_overload_hours.mean()})
network_hourly=pd.concat(network_hourly,ignore_index=True); network_summary=pd.DataFrame(network_summary)
save_table(network_hourly,'Table_24_Feeder_Hourly_Central'); save_table(network_summary,'Table_25_Feeder_Uncertainty_Summary')
note('\n8. DISTRIBUTION-FEEDER INTERACTION')
note(f'- IEEE 33-bus feeder evaluated for {len(network_policies)} policies and {N_GRID_SCENARIOS} forecast-error scenarios per policy.')
note(f'- Worst minimum voltage across policy summaries: {network_summary.min_voltage_worst.min():.3f} pu.')
note('- Results demonstrate benchmark-system interaction, not the unidentified original workplace feeder.')
fig,axes=plt.subplots(1,2,figsize=(12,4.2))
sns.lineplot(data=network_hourly,x='hour',y='min_voltage_pu',hue='policy',ax=axes[0]); axes[0].axhline(.95,color='r',ls=':')
axes[0].set(title='Central scenario minimum voltage',ylabel='Minimum voltage (pu)')
sns.barplot(data=network_summary,x='policy',y='losses_mwh_mean',ax=axes[1],color='#246A8D'); axes[1].tick_params(axis='x',rotation=20)
axes[1].set(title='Mean feeder losses under forecast uncertainty',ylabel='Daily line losses (MWh)')
fig.suptitle('Figure 10. Network consequences of charging policies',fontweight='bold')
fig.tight_layout(); save_figure(fig,'Figure_10_Feeder_Interaction'); plt.show()


## 9. Reproducibility manifest and consolidated outputs

This final cell writes:

- `outputs_summary.txt` — principal numerical results, caveats, and file locations;
- `run_configuration.json` — seed, paths, and runtime options;
- `output_inventory.csv` — all generated deliverables.

Run this cell after all desired analyses.

In [ ]:
note('\n9. INTERPRETATION BOUNDARIES')
note('- Bunching is consistent with response to the four-hour rule; it is not definitive causal identification.')
note('- The threshold result does not identify a continuous per-kWh elasticity.')
note('- Forecast load is reconstructed from session data and is not interval-metered feeder demand.')
note('- Pricing-control results are conditional simulations; grid impacts are evaluated on a standard IEEE 33-bus benchmark, not the unidentified original feeder.')
note('- Quantum results, when enabled, are exploratory simulator benchmarks; no quantum advantage is claimed.')

config={
    'generated_utc':pd.Timestamp.utcnow().isoformat(), 'data_path':str(DATA_PATH),
    'output_root':str(OUTPUT_ROOT),'seed':SEED,'n_bootstrap':N_BOOT,
    'n_agent_seeds':N_AGENT_SEEDS,'n_quantum_seeds':N_QUANTUM_SEEDS,
    'quantum_steps':QUANTUM_STEPS,'quantum_cap_per_split':QUANTUM_CAP_PER_SPLIT,
    'run_quantum':RUN_QUANTUM,'fast_mode':FAST_MODE,
    'python_version':os.sys.version
}
with open(OUTPUT_ROOT/'run_configuration.json','w') as f: json.dump(config,f,indent=2)

with open(OUTPUT_ROOT/'outputs_summary.txt','w',encoding='utf-8') as f:
    f.write('\n'.join(summary_lines)+'\n')

inventory=[]
for path in sorted(OUTPUT_ROOT.rglob('*')):
    if path.is_file():
        inventory.append({'relative_path':str(path.relative_to(OUTPUT_ROOT)),
                          'size_bytes':path.stat().st_size,'extension':path.suffix.lower()})
inventory=pd.DataFrame(inventory)
inventory.to_csv(OUTPUT_ROOT/'output_inventory.csv',index=False)

print('\nAnalysis complete.')
print('All outputs saved to:', OUTPUT_ROOT)
print('Summary:', OUTPUT_ROOT/'outputs_summary.txt')
display(inventory)


## 10. Manuscript-facing conclusions

The revised workflow supports more defensible conclusions:

1. A concentration of session durations below four hours can be quantified with session- and user-level uncertainty and challenged using specification and placebo tests.
2. User profiles can be constructed without defining them by the fee outcome later used for interpretation.
3. Unseen-user classification should exclude held-out users' outcome histories; performance must be reported under this stricter task definition.
4. Operational load forecasting requires chronological validation and a seasonal-naive comparator.
5. The pricing environment should be treated as a sensitivity analysis because continuous elasticity is not identified by the threshold data.
6. Quantum benchmarks are matched across folds, seeds, subsets and budgets, with finite-shot/noise diagnostics and no claim of advantage.
7. Q-EVCS is evaluated as a multi-seed simulated pricing controller against a matched-budget classical comparator.
8. Charging policies are translated into voltage, loading, losses, and overload metrics on an IEEE 33-bus benchmark with forecast-error propagation.

### Remaining IJEPES requirement

The notebook now demonstrates system-level interaction on a standard benchmark feeder. The limitation to preserve in the manuscript is that the original workplace feeder topology and interval measurements are unavailable; therefore, network conclusions demonstrate methodological capability and scenario robustness rather than site-specific field validation.